In [ ]:
from shutil import copytree

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

from study.data import cohort, features, prediction_matrix, prediction_metadata
from study.features import DYNAMIC_VARS
from study.io import RESOURCES, ROOT, WORK, load, reset_directory
from study.metrics import probability_metrics
from study.models import calibration_summary
from study.plotting import save_table
from study.statistics import (
    loss_matrix,
    mean_draw_losses,
    paired_auc_interval,
    paired_bootstrap,
    paired_difference,
    risk_matrix,
)

In [ ]:
PAIRS = [
    ('process_update', 'R3_raw'),
    ('process_update', 'R2_raw'),
    ('process_update', 'R3_platt'),
    ('process_update', 'R3_intercept'),
    ('process_update', 'unrestricted_stack'),
    ('unrestricted_stack', 'R3_platt')
]

SETTINGS = [
    ('carevue', 'carevue', 'original'),
    ('carevue', 'metavision', 'original'),
    ('metavision', 'mimiciv', 'shared'),
    ('metavision', 'eicu', 'shared')
]

OUTPUT = ROOT / 'results/accompanying'

In [ ]:
reset_directory(OUTPUT)

source_fits = load(WORK / 'models/source.pkl')

full = pd.read_parquet(WORK / 'predictions/full')
physiology = pd.read_parquet(WORK / 'results/physiology.parquet')
observation = pd.read_parquet(WORK / 'results/observation.parquet')

In [ ]:
# Summarise full input performance and calibration
rows = []

for (source, domain, support, tier, repeat), group in full.groupby(
    ['source', 'domain', 'support', 'tier', 'repeat'], observed = True
):
    group = group.loc[group.partition.eq('test')] if domain == 'eicu' else group
    y, risk = (group.in_hospital_mortality.to_numpy(), group.risk.to_numpy())
    calibration = calibration_summary(y, risk)
    alerts = np.argsort(-risk, kind = 'stable')[: max(1, int(np.ceil(0.1 * len(group))))]

    rows.append(
        dict(
            source = source,
            domain = domain,
            support = support,
            repeat = repeat,
            tier = tier,
            **probability_metrics(y, risk),
            sensitivity_at_10_percent = y[alerts].sum() / y.sum(),
            **calibration,
        )
    )

per_allocation = pd.DataFrame(rows)
performance = (
    per_allocation.groupby(['source', 'domain', 'support', 'tier'], observed = True)
    .mean(numeric_only = True)
    .drop(columns = 'repeat')
    .reset_index()
)
performance['evaluation'] = np.select(
    [performance.source.eq(performance.domain)],
    ['source_oof'],
    default = 'target_test',
)

save_table(performance, 'full_performance', directory = OUTPUT)

In [ ]:
# Export complete supporting results
for name, result in [
    ('learner_comparisons', 'learners'),
    ('source_refit_summary', 'source_refit_summary'),
    ('timing_comparisons', 'timing'),
    ('timing_interactions', 'timing_interactions'),
    ('hospital_thresholds', 'hospital_thresholds'),
    ('ventilation_sensitivity', 'ventilation'),
    ('reverse_transfer', 'reciprocal'),
    ('local_availability', 'local_availability'),
    ('local_recalibration', 'local_calibration'),
    ('hospital_effects', 'hospital_effects')
]:
    frame = pd.read_parquet(WORK / 'results' / f'{result}.parquet')
    if result == 'local_calibration':
        frame['uses_target_calibration_labels'] = frame.calibration.ne('raw')
    
    save_table(frame, name, directory = OUTPUT)

contrasts = pd.concat([physiology, observation], ignore_index = True)

save_table(contrasts, 'full_input_comparisons', directory = OUTPUT)

In [ ]:
# Compare pooled updates
rows = []
for condition in ('full', 'retain_0.5', 'retain_0.25'):
    metadata = cohort('eicu').query("partition == 'test'").sort_index()
    ids = metadata.index

    for candidate, reference in PAIRS:
        filters = [('restriction', '==', condition)]
        left = prediction_matrix('pooled', ids, filters + [('method', '==', candidate)])
        right = prediction_matrix('pooled', ids, filters + [('method', '==', reference)])
        left, right = (left.clip(1e-06, 1 - 1e-06), right.clip(1e-06, 1 - 1e-06))

        difference = mean_draw_losses(loss_matrix(metadata.in_hospital_mortality, left)) - mean_draw_losses(
            loss_matrix(metadata.in_hospital_mortality, right)
        )
        sampled = paired_bootstrap(metadata, difference, 5000, 77, cluster = True)
        points, draws = (sampled['points'], sampled['draws'])
        lower, upper = np.quantile(draws, [0.025, 0.975])

        p_value = min(
            1.0, 2 * min((np.sum(draws <= 0) + 1) / (len(draws) + 1), (np.sum(draws >= 0) + 1) / (len(draws) + 1))
        )

        rows.append(
            dict(
                restriction = condition,
                candidate = candidate,
                reference = reference,
                difference = points.mean(),
                lower = lower,
                upper = upper,
                p_value = p_value,
                allocations = len(points),
                allocation_minimum = points.min(),
                allocation_maximum = points.max(),
                allocation_fraction_below_zero = points.lt(0).mean(),
            )
        )

pooled_contrasts = pd.DataFrame(rows)
pooled_contrasts['holm_p'] = multipletests(pooled_contrasts.p_value, method = 'holm')[1]
save_table(
    pooled_contrasts.assign(analysis_role = 'supporting_target_adaptation', uses_target_calibration_labels = True),
    'pooled_update_comparisons',
    directory = OUTPUT,
)

coefficients = pd.read_parquet(WORK / 'results/pooled_coefficients.parquet')
weights = (
    coefficients.groupby(['restriction', 'repeat'], observed = True)
    .gamma.mean()
    .groupby('restriction', observed = True)
    .mean()
    .rename('gamma')
    .reset_index()
)

save_table(weights, 'pooled_process_weights', directory = OUTPUT)

In [ ]:
# Export recording summaries
rows = []
for domain in ('carevue', 'metavision', 'mimiciv', 'eicu'):
    metadata, x = (cohort(domain), features(domain))
    ids = metadata.index[metadata.partition.eq('test')] if domain == 'eicu' else metadata.index

    # Summarise recording coverage, counts, and last measurement time
    for variable in [name for name in DYNAMIC_VARS if name != 'MechVent']:
        count = x.loc[ids, variable + '_count']
        observed = count.gt(0)
        rows.append(
            dict(
                domain = domain,
                variable = variable,
                fraction = observed.mean(),
                median_count = count.loc[observed].median(),
                last_hour = x.loc[ids, variable + '_last_measurement_hour'].loc[observed].median(),
            )
        )

recording = pd.DataFrame(rows)

save_table(recording, 'common_variable_recording', directory = OUTPUT)

In [ ]:
# Frozen source-only calibration
calibrated = pd.read_parquet(WORK / 'predictions/full_source_calibrated')
metric_rows, comparison_rows = [], []

for domain in ('mimiciv', 'eicu'):
    metadata = cohort(domain).sort_index()

    if domain == 'eicu':
        metadata = metadata.loc[metadata.partition.eq('test')]

    metadata = prediction_metadata(metadata)
    group = calibrated.loc[calibrated.domain.eq(domain) & calibrated.support.eq('shared')]

    # Summarise target performance after source-only Platt correction
    for tier in ('R0', 'R1', 'R2', 'R3', 'R4'):
        risk = risk_matrix(group.loc[group.tier.eq(tier)], metadata.index).iloc[:, 0]
        metric_rows.append(
            dict(
                domain = domain,
                support = 'shared',
                tier = tier,
                calibration = 'source_oof_platt',
                **probability_metrics(metadata.in_hospital_mortality, risk),
                **calibration_summary(metadata.in_hospital_mortality, risk),
            )
        )

    # Pair corrected tiers on the same patients
    for richer, simpler in [('R2', 'R1'), ('R3', 'R2')]:
        candidate = risk_matrix(group.loc[group.tier.eq(richer)], metadata.index)
        reference = risk_matrix(group.loc[group.tier.eq(simpler)], metadata.index)

        for equal_hospital in [False, True] if domain == 'eicu' else [False]:
            result = paired_difference(
                metadata, candidate, reference, cluster = domain == 'eicu', equal_hospital = equal_hospital
            )

            if not equal_hospital:
                result.update(paired_auc_interval(metadata, candidate, reference, domain == 'eicu', 2000, 77))

            comparison_rows.append(
                dict(
                    domain = domain, contrast = richer + '_minus_' + simpler,
                    calibration = 'source_oof_platt', **result,
                )
            )

save_table(pd.DataFrame(metric_rows), 'source_oof_calibrated_performance', directory = OUTPUT)
save_table(pd.DataFrame(comparison_rows), 'source_oof_calibrated_increments', directory = OUTPUT)

In [ ]:
# Include the canonical feature definitions, mappings, and restriction definitions
copytree(RESOURCES, OUTPUT / 'resources')